In [14]:

from typing import Dict, Tuple
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import MNIST
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np

%matplotlib inline

from torch import autograd
from torch.autograd import Variable
from tensorboardX import SummaryWriter
import torch.optim as optim
import torchvision.datasets as datasets
import time
import os


if __name__ == "__main__":
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "No GPU detected")

Torch version: 2.7.0+cu126
CUDA available: True
CUDA version: 12.6
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 4090


In [15]:
def plot_shape(shape_matrix):
    """Plot the generated shape (expects input shape (1, 32, 32) or (32, 32))."""
    # Squeeze channel if present
    if shape_matrix.ndim == 3 and shape_matrix.shape[0] == 1:
        shape_matrix = shape_matrix.squeeze(0)  # → (32, 32)

    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 6))
    ax.set_facecolor('#301934')
    ax.imshow(shape_matrix, origin='upper', cmap='viridis')  # add colormap if needed
    plt.axis('off')
    # print(f'size: {shape_matrix.shape[0]} x {shape_matrix.shape[1]}')
    plt.show()


def load_item(item, p= True, action=''):
    if action=='':
        if p:
            print(f'Cond: {item[0]}')
            print(f'Params: {item[1]}')
        plot_shape(item[2])
        return {'Cond':item[0], 'Params':item[1]}
    if action == 'shape':
        return item[3]
    
def quarter(matrix):
    return matrix[:32, :32]

In [16]:
class SelfAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.q = nn.Conv2d(in_channels, in_channels, 1)
        self.k = nn.Conv2d(in_channels, in_channels, 1)
        self.v = nn.Conv2d(in_channels, in_channels, 1)
        self.proj = nn.Conv2d(in_channels, in_channels, 1)
    def forward(self, x):
        B, C, H, W = x.shape
        q = self.q(x).reshape(B, C, -1)
        k = self.k(x).reshape(B, C, -1)
        v = self.v(x).reshape(B, C, -1)
        attn = torch.softmax(q.transpose(1,2) @ k / (C**0.5), dim=-1)
        out = (attn @ v.transpose(1,2)).transpose(1,2).reshape(B, C, H, W)
        return self.proj(out) + x

In [17]:
class AdaIN(nn.Module):
    def __init__(self, channels, cond_dim):
        super().__init__()
        self.fc = nn.Linear(cond_dim, channels*2)
    def forward(self, x, cond):
        h = self.fc(cond)
        gamma, beta = h.chunk(2, dim=1)
        gamma = gamma.unsqueeze(-1).unsqueeze(-1)
        beta = beta.unsqueeze(-1).unsqueeze(-1)
        mean = x.mean([2,3], keepdim=True)
        std = x.std([2,3], keepdim=True)
        x_norm = (x - mean) / (std + 1e-5)
        return gamma * x_norm + beta

In [18]:
def get_timestep_embedding(timesteps, embedding_dim):
    # Sinusoidal positional encoding
    half_dim = embedding_dim // 2
    emb = np.log(10000) / (half_dim - 1)
    emb = torch.exp(torch.arange(half_dim, dtype=torch.float32, device=timesteps.device) * -emb)
    emb = timesteps.float().unsqueeze(1) * emb.unsqueeze(0)
    emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
    if embedding_dim % 2 == 1:  # zero pad
        emb = F.pad(emb, (0,1,0,0))
    return emb

In [19]:
class ImprovedResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, cond_dim=8, use_attention=False):
        super().__init__()
        self.same_channels = in_channels == out_channels
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, 1, 1)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.ada = AdaIN(out_channels, cond_dim)
        self.use_attention = use_attention
        if use_attention:
            self.attn = SelfAttention(out_channels)
        else:
            self.attn = nn.Identity()
    def forward(self, x, cond):
        h = F.gelu(self.norm1(self.conv1(x)))
        h = self.ada(h, cond)
        h = F.gelu(self.norm2(self.conv2(h)))
        h = self.attn(h)
        if self.same_channels:
            return (x + h) / 1.414
        else:
            return h

In [28]:
class UnetDown(nn.Module):
    """
    Down-sampling block that applies a ResBlock with conditioning, then 2×2 max-pool.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = ImprovedResBlock(in_channels, out_channels)
        self.pool  = nn.MaxPool2d(2)

    def forward(self, x, cond):
        x = self.block(x, cond)   # pass both inputs
        x = self.pool(x)
        return x



In [21]:
class UnetUp(nn.Module):
    """
    Upsample → concat with skip → two ResBlocks.

    • __init__(in_channels, out_channels)          –– unchanged
    • forward(x, skip)                             –– unchanged signature
        x    : feature map from previous decoder stage
        skip : same-resolution feature map from encoder
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.out_channels = out_channels

        # 1× up-convolution (doubles H,W, sets channels → out_channels)
        self.up = nn.ConvTranspose2d(in_channels, out_channels,
                                     kernel_size=2, stride=2)

        # We can’t know skip-channels at construction time,
        # so build the first ResBlock lazily on the first forward pass.
        self.block1 = None           # will become ImprovedResBlock(...)
        self.block2 = ImprovedResBlock(out_channels, out_channels)

    def _build_block1(self, in_ch: int, device: torch.device):
        """Create the first ResBlock once we know concat-channel count."""
        self.block1 = ImprovedResBlock(in_ch, self.out_channels).to(device)

    def forward(self, x: torch.Tensor, skip: torch.Tensor, cond=None):
        """
        Args
        ----
        x    : (B,   in_channels,  H/2, W/2)
        skip : (B, skip_channels,  H,   W)
        """
        x = self.up(x)               # → (B, out_channels, H, W)
        x = torch.cat([x, skip], dim=1)  # concat adds skip_channels

        # Build block1 the first time, now that we know concat size
        if self.block1 is None:
            self._build_block1(x.shape[1], x.device)

        x = self.block1(x, cond)
        x = self.block2(x, cond)
        return x

# a = torch.empty(32, 2, 16,16)
# b = UnetUp(4,1)
# print(b.forward(a,a).size())

In [22]:
class EmbedFC(nn.Module):
    """
    Use FC layer for embedding 1-d metadata, like modes+weights
    (putting into higher dimension)
    Effectively our conditional
    input: Conditional, size (batchsize, input_dim = 4+4)
    Output: Higherdimensional tensor, size (batchsize, output_dim)
    
    """
    def __init__(self, input_dim, emb_dim):
        super(EmbedFC, self).__init__()
        '''
        generic one layer FC NN for embedding things  
        '''
        self.input_dim = input_dim
        layers = [
            nn.Linear(input_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim),
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(-1, self.input_dim)
        return self.model(x)

a = torch.empty(32, 8)
b = EmbedFC(8, 32)
print(b.forward(a).size())

torch.Size([32, 32])


In [30]:
class ContextUnet(nn.Module):
    """
    U-Net style neural network for conditional image generation, conditioning on
    both timestep t and a context vector c (modes+weights)

    ** Diverges from MNIST Example:
        - instead of inputting n_classes, input the length of my
          conditional, as is continuous 8
    """
    def __init__(self, in_channels, n_feat = 256, cond_dim=8):
        """
        in_channels (1 for greyscale), n_feat is base feature size, n_classes is number of labels
        """
        super(ContextUnet, self).__init__()

        self.in_channels = in_channels
        self.n_feat = n_feat
        self.cond_dim = cond_dim

        # Lifts image channels to n_feat using our residual block
        self.init_conv = ImprovedResBlock(in_channels, n_feat)

        self.down1 = UnetDown(n_feat, n_feat) # (batchsize, 1, 32, 32) -> (batchsize, 2, 16, 16)
        self.down2 = UnetDown(n_feat, 2 * n_feat) # (batchsize, 2, 16, 16) -> (batchsize, 4, 8, 8)

        # reduces 2d feature map down2 into small latent vector of shape
        # (batchsize, 2*n_feat, 1, 1) via 7x7 average pooling
        self.to_vec = nn.Sequential(nn.AvgPool2d(8), nn.GELU())

        # Embeds timestep t and context c into vectors that will later be reshaped and added
        # to upsampling path
        self.timeembed1 = EmbedFC(1, 2*n_feat)
        self.timeembed2 = EmbedFC(1, 1*n_feat)
        self.contextembed1 = EmbedFC(cond_dim, 2*n_feat)
        self.contextembed2 = EmbedFC(cond_dim, 1*n_feat)

        # upsamples latent vector (hiddenvec) back to 2d spatial mapping
        self.up0 = nn.Sequential(
            # nn.ConvTranspose2d(6 * n_feat, 2 * n_feat, 7, 7), # when concat temb and cemb end up w 6*n_feat
            nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, 8, 8), # otherwise just have 2*n_feat
            nn.GroupNorm(8, 2 * n_feat),
            nn.ReLU(),
        )
        # upsampling (decoder) blocks that fuse upsampled features with skip connections from
        # the encoder
        self.up1 = UnetUp(2 * n_feat, n_feat)
        self.up2 = UnetUp(2 * n_feat, n_feat)

        # final processing layer to reduce features back to the original number of channels
        # Ideally produces the final denoised image!
        self.out = nn.Sequential(
            nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1),
            nn.GroupNorm(8, n_feat),
            nn.ReLU(),
            nn.Conv2d(n_feat, self.in_channels, 3, 1, 1),
        )

    def forward(self, x, c, t):
        # x is (noisy) image, size (batchsize, in_channels, 32, 32)
        # c is context label,size (batchsize, 8)
        # t is timestep scalar, size (batchsize, 1)
        # Binary mask for whether to apply conditioning 
        # probably will not need because need conditioning

        x = self.init_conv(x, c)
        down1 = self.down1(x, c)
        down2 = self.down2(down1, c)
        hiddenvec = self.to_vec(down2) # pooled latent representation

        # convert context to one hot embedding
        
        # embed context, time step, reshapes them for broadcasting in upsampling layers
        cemb1 = self.contextembed1(c).view(-1, self.n_feat * 2, 1, 1)
        temb1 = self.timeembed1(t).view(-1, self.n_feat * 2, 1, 1)
        cemb2 = self.contextembed2(c).view(-1, self.n_feat, 1, 1)
        temb2 = self.timeembed2(t).view(-1, self.n_feat, 1, 1)

        # could concatenate the context embedding here instead of adaGN
        # hiddenvec = torch.cat((hiddenvec, temb1, cemb1), 1)

        up1 = self.up0(hiddenvec)
        # up2 = self.up1(up1, down2) # if want to avoid add and multiply embeddings
        up2 = self.up1(cemb1*up1+ temb1, down2)  # add and multiply embeddings
        up3 = self.up2(cemb2*up2+ temb2, down1)
        out = self.out(torch.cat((up3, x), 1))
        return out

model = ContextUnet(1, 64, 8)
x = torch.zeros(32, 1, 32, 32)           # batch of noisy grayscale images
c = torch.zeros(32, 8)          # context labels
t = torch.rand(32, 1)                    # timestep in [0, 1]

output = model.forward(x, c, t)
print(output.shape)



RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 16 but got size 8 for tensor number 1 in the list.

In [31]:
class ImprovedUNet(nn.Module):
    def __init__(self, in_channels=1, base=64, cond_dim=8, time_dim=128):
        super().__init__()
        self.time_dim = time_dim
        self.time_embed = nn.Linear(time_dim, cond_dim)
        # Down
        self.enc1 = ImprovedResBlock(in_channels, base, cond_dim, use_attention=False)
        self.enc2 = ImprovedResBlock(base, base*2, cond_dim, use_attention=True)
        self.enc3 = ImprovedResBlock(base*2, base*4, cond_dim, use_attention=True)
        self.enc4 = ImprovedResBlock(base*4, base*8, cond_dim, use_attention=True)
        # Up
        self.up1 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.dec1 = ImprovedResBlock(base*8, base*4, cond_dim, use_attention=True)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.dec2 = ImprovedResBlock(base*4, base*2, cond_dim, use_attention=True)
        self.up3 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.dec3 = ImprovedResBlock(base*2, base, cond_dim, use_attention=False)
        self.out = nn.Conv2d(base, in_channels, 1)
    def forward(self, x, cond, t):
        t_emb = get_timestep_embedding(t, self.time_dim).to(x.device)
        cond = cond + self.time_embed(t_emb)
        e1 = self.enc1(x, cond)
        e2 = self.enc2(F.avg_pool2d(e1, 2), cond)
        e3 = self.enc3(F.avg_pool2d(e2, 2), cond)
        e4 = self.enc4(F.avg_pool2d(e3, 2), cond)
        d1 = self.up1(e4)
        d1 = torch.cat([d1, e3], 1)
        d1 = self.dec1(d1, cond)
        d2 = self.up2(d1)
        d2 = torch.cat([d2, e2], 1)
        d2 = self.dec2(d2, cond)
        d3 = self.up3(d2)
        d3 = torch.cat([d3, e1], 1)
        d3 = self.dec3(d3, cond)
        return self.out(d3)

In [ ]:
def ddpm_schedules(beta1, beta2, T):
    """
    Precomputes all noise scheduling terms needed for training and sampling
    from a denoising diffusion probabilistic model
    Uses a sequence of gradually increasing noise level over T timesteps
    beta1: starting noise level, O(1e-4)
    beta2: final noise level, O(0.02)
    T: number of time steps
    """
    assert beta1 < beta2 < 1.0, "beta1 and beta2 must be in (0, 1)"

    beta_t = (beta2 - beta1) * torch.arange(0, T + 1, dtype=torch.float32) / T + beta1 # noise variance schedule (for every time t in T)
    sqrt_beta_t = torch.sqrt(beta_t)
    alpha_t = 1 - beta_t
    log_alpha_t = torch.log(alpha_t)
    alphabar_t = torch.cumsum(log_alpha_t, dim=0).exp()

    sqrtab = torch.sqrt(alphabar_t)
    oneover_sqrta = 1 / torch.sqrt(alpha_t)

    sqrtmab = torch.sqrt(1 - alphabar_t)
    mab_over_sqrtmab_inv = (1 - alpha_t) / sqrtmab

    # dictionary of schedule terms
    return {
        "alpha_t": alpha_t,  # \alpha_t , signal retention at time step t
        "oneover_sqrta": oneover_sqrta,  # 1/\sqrt{\alpha_t}
        "sqrt_beta_t": sqrt_beta_t,  # \sqrt{\beta_t} , noise scaling factor
        "alphabar_t": alphabar_t,  # \bar{\alpha_t} , cumulative signal retention
        "sqrtab": sqrtab,  # \sqrt{\bar{\alpha_t}} , scales clean image during noise
        "sqrtmab": sqrtmab,  # \sqrt{1-\bar{\alpha_t}} , noise strength
        "mab_over_sqrtmab": mab_over_sqrtmab_inv,  # (1-\alpha_t)/\sqrt{1-\bar{\alpha_t}} , for reverse diffusion
    }


In [ ]:

class DDPM(nn.Module):
    """
    Denoising Diffusion Probabilistic Model
    """
    def __init__(self, nn_model, betas, n_T, device, drop_prob=0.1):
        """
        betas: tuple (beta1, beta2) for linear noise schedule
        n_T: total number of diffusion steps (e.g. 1000)
        drop_prob: probability of dropping conditioning (for classifier free guidance)
        """
        super(DDPM, self).__init__()
        self.nn_model = nn_model.to(device)

        # register_buffer allows accessing dictionary produced by ddpm_schedules
        # e.g. can access self.sqrtab later
        for k, v in ddpm_schedules(betas[0], betas[1], n_T).items():
            self.register_buffer(k, v)

        self.n_T = n_T
        self.device = device
        self.drop_prob = drop_prob
        self.loss_mse = nn.MSELoss()

    def forward(self, x, c):
        """
        x: clean image tensor, size (batchsize, 1, 32, 32)
        c: conditional vector, size (batchsize, 32)
        this method is used in training, so samples t and noise randomly
        """

        # t ~ Uniform(0, n_T)
        # sample a random timestep t for each item in the batch, 
        # determines how much noise to add
        _ts = torch.randint(1, self.n_T+1, (x.shape[0],)).to(self.device)
        noise = torch.randn_like(x)  # eps ~ N(0, 1)

        # Generates noising image x_t from clean image x
        x_t = (
            self.sqrtab[_ts, None, None, None] * x
            + self.sqrtmab[_ts, None, None, None] * noise
        )  # This is the x_t, which is sqrt(alphabar) x_0 + sqrt(1-alphabar) * eps
        # We should predict the "error term" from this x_t. Loss is what we return.

        # dropout context with some probability
        # context_mask = torch.bernoulli(
        #     torch.zeros_like(c)+self.drop_prob).to(self.device)

        # return MSE between added noise, and our predicted noise
        # runs x_t, c, t, and context_mask through model, compares predicted noise
        # with actual noise using MSE
        # return self.loss_mse(noise, self.nn_model(x_t, c, _ts / self.n_T, context_mask))
        return self.loss_mse(noise, self.nn_model(x_t, c, _ts / self.n_T))

    def sample(self, n_sample, size, device, c_i, guide_w=0.0):
        """
        n_sample: number of images to generate
        size: shape of each image, [1,32,32]
        guid_w: guidance strength, 0=no guidance, >0=stronger conditioning (what we want)
        """
        # we follow the guidance sampling scheme described in 'Classifier-Free Diffusion Guidance'
        # to make the fwd passes efficient, we concat two versions of the dataset,
        # one with context_mask=0 and the other context_mask=1
        # we then mix the outputs with the guidance scale, w
        # where w>0 means more guidance

        # x_T ~ N(0, 1), sample initial noise
        x_i = torch.randn(n_sample, *size).to(device)  # start from pure noise
        # context for us just cycles throught the mnist labels

        x_i_store = []  # keep track of generated steps in case want to plot something
        print()
        # Iterate over timesteps in revers (from noise -> image)
        for i in range(self.n_T, 0, -1):
            print(f'sampling timestep {i}', end='\r')
            t_is = torch.tensor([i / self.n_T], device=device).repeat(n_sample, 1)

            z = torch.randn(n_sample, *size).to(device) if i > 1 else 0 # add noise at all steps except final one

            # predict the noise using both conditioned and unconditioned branches
            eps = self.nn_model(x_i, c_i, t_is)
            # apply classifier-free guidance formula: ϵ = (1 + w)⋅ϵ_cond − w⋅ϵ_uncond

            x_i = (
                self.oneover_sqrta[i] * (x_i - eps * self.mab_over_sqrtmab[i])
                + self.sqrt_beta_t[i] * z
            )
            # save frames for visualization every 20 steps and near the end
            if i % 20 == 0 or i == self.n_T or i < 8:
                x_i_store.append(x_i.detach().cpu().numpy())

        # returns final denoised image x_i and intermediate steps x_i_store
        x_i_store = np.array(x_i_store)
        return x_i, x_i_store

ddpm = DDPM(model, betas=(1e-4, 0.02), n_T=10, device='cpu')
x = torch.zeros(4, 1, 32, 32)
c = torch.zeros(4, 8)
loss = ddpm.forward(x, c)
print(loss)

samples, history = ddpm.sample(n_sample=4, size=(1, 32, 32), device='cpu', c_i=c, guide_w=2.0)
print(samples)

tensor(1.0389, grad_fn=<MseLossBackward0>)

tensor([[[[-0.7897,  0.2155, -0.9513,  ..., -0.2175,  0.9745,  0.2132],
          [ 0.5704,  0.1321,  0.4644,  ..., -0.5048, -0.7210, -0.0966],
          [-0.2413, -0.2335,  0.9755,  ...,  0.8426,  1.5662,  0.5954],
          ...,
          [-0.0589,  0.4735,  0.1447,  ..., -0.4210, -0.4143,  0.3373],
          [-0.5843, -0.9405, -1.0360,  ..., -0.1299,  1.2425,  0.2113],
          [-1.2073,  0.7529,  0.8551,  ...,  1.2865, -2.8929,  1.9422]]],


        [[[ 0.4805,  0.2401,  0.2414,  ...,  1.9164,  2.0606, -0.9552],
          [-0.3361, -0.4411, -0.6095,  ...,  1.1697, -0.6989,  1.1723],
          [ 1.5456, -1.3877,  4.0376,  ...,  0.0179, -2.1394,  0.7060],
          ...,
          [ 0.6311, -0.8457,  0.0927,  ..., -0.8931,  2.3995,  0.5707],
          [-0.8955,  0.2709,  1.6082,  ...,  0.4114,  1.0603,  1.2033],
          [-0.4156, -1.4436, -0.6867,  ..., -1.0093,  2.2768,  0.9144]]],


        [[[-0.5244, -0.1193,  1.0733,  ...,  0.6633,  

In [ ]:
# n_epoch = 20
# batch_size = 32
# n_T = 400
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# n_feat = 128
# lrate = 1e-4
# save_model = True
save_dir = './data/waveguide_diffusion_outputs/'
# guide_weights = [0.0, 0.5, 2.0]  # guidance scaling

os.makedirs(save_dir, exist_ok=True)

# # Define the model
# model = ContextUnet(in_channels=1, n_feat=n_feat, cond_dim=8)  # cond_dim = 4 eig + 4 wts
# ddpm = DDPM(nn_model=model, betas=(1e-4, 0.02), n_T=n_T, device=device, drop_prob=0.1)
# loss = ddpm.sample(x,c)

In [ ]:
# dataset = WaveguideDataset('train_test_split.h5')
from waveguide_dataset import WaveguideDataset
dataset = WaveguideDataset('train_test_split.h5')
# dataloader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=4)
# #dataloader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=5)
# pbar = tqdm(dataloader)
# i=0
# for c, p, x in pbar:
#    i+=1
# print(i)



In [ ]:
def train_waveguide(dataset):

    # hardcoding these here
    n_epoch = 20
    batch_size = 256
    n_T = 400 # difusion steps
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    # WILL HAVE TO CHANGE THIS to cond_dim = 8
    cond_dim = 8

    n_feat = 128 # Feature size for U-Net, may have to make 256
    lrate = 1e-4
    save_model = True
    save_dir = './data/diffusion_v1_outputs_log_norm/'
    os.makedirs(save_dir, exist_ok=True)

    ws_test = [0.0, 0.5, 1.0, 2.0] # strength of generative guidance

    # FOR DISPLAY CHECK
    ########################################
    n_sample = 40  # 5x8, or whatever number you want
    condition_list = []
    x_real = []
    # Pull first n_sample items from dataset, 
    for i in range(n_sample):
        cond_i, _, x_i = dataset[i]  # cond: (8,), x_i: (1, 32, 32)
        condition_list.append(cond_i)
        x_real.append(x_i)
    c_i = torch.stack(condition_list).to(device)  # (n_sample, 8)
    x_real = torch.stack(x_real).to(device) 
    ########################################

    ddpm = DDPM(nn_model=ContextUnet(in_channels=1, n_feat=n_feat, cond_dim=cond_dim), betas=(1e-4, 0.02), n_T=n_T, device=device, drop_prob=0.1)

    ddpm.to(device)

    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=5)

    optim = torch.optim.Adam(ddpm.parameters(), lr=lrate) # ADAM optimizer for training, keep

    for ep in range(n_epoch):
        print(f'epoch {ep}')
        ddpm.train()

        # linear lrate decay, so training quicker at begining, more fine-tuned at end?
        optim.param_groups[0]['lr'] = lrate*(1-ep/n_epoch)

        # batch training
        pbar = tqdm(dataloader)
        loss_ema = None # For display

        for c, p, x in pbar: # loops over batches from dataset
            optim.zero_grad()
            x = x.to(device)
            c = c.to(device)
            loss = ddpm(x, c) # predicts noise from noisy image
            loss.backward() # Computes EMA-smoothed loss for live progress display, backprops to update weights

            if loss_ema is None:
                loss_ema = loss.item()
            else:
                loss_ema = 0.95 * loss_ema + 0.05 * loss.item() 
            pbar.set_description(f"loss: {loss_ema:.4f}")
            optim.step()
        
        # for eval, save an image of currently generated samples (top rows)
        # followed by real images (bottom rows)
        ddpm.eval() # evaluates model without gradient
        with torch.no_grad():
            for w_i, w in enumerate(ws_test):
                # WILL HAVE TO CHANGE TO MAKE WORK WITH MY SAMPLING STRATEGY, ie grabbing first 40 waveguides and generating new ones
                x_gen, x_gen_store = ddpm.sample(n_sample, (1, 32, 32), device, c_i=c_i, guide_w=w) # samples n_sample generated images, 

                # append some real images at bottom, order by class also
                # x_real = torch.Tensor(x_gen.shape).to(device)
                # for k in range(n_classes):
                #     for j in range(int(n_sample/n_classes)):
                #         try: 
                #             idx = torch.squeeze((c == k).nonzero())[j]
                #         except:
                #             idx = 0
                #         x_real[k+(j*n_classes)] = x[idx]

                x_all = torch.cat([x_gen, x_real], dim=0)
                grid = make_grid(x_all*-1 + 1, nrow=8)
                save_image(grid, save_dir + f"image_ep{ep}_w{w}.png")
                print('saved image at ' + save_dir + f"image_ep{ep}_w{w}.png")

                if ep%5==0 or ep == int(n_epoch-1):
                    # create gif of images evolving over time, based on x_gen_store
                    fig, axs = plt.subplots(nrows=n_sample // 8, ncols=8,sharex=True,sharey=True,figsize=(8,3))

                    def animate_diff(i, x_gen_store):
                        print(f'gif animating frame {i} of {x_gen_store.shape[0]}', end='\r')
                        plots = []
                        for row in range(int(n_sample // 8)):
                            for col in range(8):
                                axs[row, col].clear()
                                axs[row, col].set_xticks([])
                                axs[row, col].set_yticks([])
                                # plots.append(axs[row, col].imshow(x_gen_store[i,(row*n_classes)+col,0],cmap='gray'))
                                img = -x_gen_store[i, (row*8) + col, 0]
                                plots.append(axs[row,col].imshow(img, cmap='gray', vmin=img.min(), vmax=img.max()))
                                # plots.append(axs[row, col].imshow(-x_gen_store[i,(row*n_classes)+col,0],cmap='gray',vmin=(-x_gen_store[i]).min(), vmax=(-x_gen_store[i]).max()))
                        return plots

                    ani = FuncAnimation(fig, animate_diff, fargs=[x_gen_store],  interval=200, blit=False, repeat=True, frames=x_gen_store.shape[0])    
                    ani.save(save_dir + f"gif_ep{ep}_w{w}.gif", dpi=100, writer=PillowWriter(fps=5))
                    print('saved image at ' + save_dir + f"gif_ep{ep}_w{w}.gif")
        # optionally save model

        if save_model and ep == int(n_epoch-1):
            torch.save(ddpm.state_dict(), save_dir + f"model_{ep}.pth")
            print('saved model at ' + save_dir + f"model_{ep}.pth")


In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.utils import make_grid, save_image
from tqdm import tqdm
from matplotlib.animation import FuncAnimation, PillowWriter

def train_waveguide2(dataset):
    # Hyperparameters
    n_epoch = 50
    batch_size = 256
    n_T = 1000
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    cond_dim = 8
    n_feat = 256
    lrate = 5e-5
    save_model = True
    save_dir = './data/diffusion_v3_improved_reisdual_block/'
    os.makedirs(save_dir, exist_ok=True)

    ws_test = [0.0, 0.5, 1.0, 2.0]
    n_sample = 40

    # Setup visualization batch
    condition_list, x_real = [], []
    for i in range(n_sample):
        cond_i, _, x_i = dataset[i]
        condition_list.append(cond_i)
        x_real.append(x_i)
    c_i = torch.stack(condition_list).to(device)
    x_real = torch.stack(x_real).to(device)

    # Initialize model + optimizer
    ddpm = DDPM(
        nn_model=ContextUnet(in_channels=1, n_feat=n_feat, cond_dim=cond_dim),
        betas=(1e-4, 0.03),
        n_T=n_T,
        device=device,
        drop_prob=0.1
    ).to(device)

    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=5)
    optimizer = torch.optim.Adam(ddpm.parameters(), lr=lrate)

    # Track loss per epoch
    epoch_losses = []

    for ep in range(n_epoch):
        print(f'Epoch {ep+1}/{n_epoch}')
        ddpm.train()
        optimizer.param_groups[0]['lr'] = lrate * (1 - ep / n_epoch)

        total_loss = 0
        with tqdm(dataloader) as pbar:
            for c, p, x in pbar:
                x, c = x.to(device), c.to(device)
                optimizer.zero_grad()
                loss = ddpm(x, c)
                loss.backward()
                optimizer.step()

                total_loss += loss.item() * x.size(0)
                pbar.set_description(f"loss: {loss.item():.4f}")

        epoch_avg_loss = total_loss / len(dataloader.dataset)
        epoch_losses.append(epoch_avg_loss)

        # Save images/gif only on final epoch
        if ep == n_epoch - 1:
            ddpm.eval()
            with torch.no_grad():
                for w in ws_test:
                    x_gen, x_gen_store = ddpm.sample(
                        n_sample=n_sample,
                        size=(1, 32, 32),
                        device=device,
                        c_i=c_i,
                        guide_w=w
                    )

                    x_all = torch.cat([x_gen, x_real], dim=0)
                    grid = make_grid(x_all * -1 + 1, nrow=8)
                    img_path = os.path.join(save_dir, f"image_ep{ep}_w{w}.png")
                    save_image(grid, img_path)
                    print(f'Saved image at {img_path}')

                    # GIF generation
                    fig, axs = plt.subplots(nrows=n_sample // 8, ncols=8, figsize=(8, 3), sharex=True, sharey=True)

                    def animate_diff(i, x_gen_store):
                        plots = []
                        for row in range(n_sample // 8):
                            for col in range(8):
                                axs[row, col].clear()
                                axs[row, col].set_xticks([])
                                axs[row, col].set_yticks([])
                                img = -x_gen_store[i, (row * 8) + col, 0]
                                plots.append(axs[row, col].imshow(img, cmap='gray', vmin=img.min(), vmax=img.max()))
                        return plots

                    ani = FuncAnimation(fig, animate_diff, fargs=[x_gen_store],
                                        interval=200, blit=False, repeat=True,
                                        frames=x_gen_store.shape[0])
                    gif_path = os.path.join(save_dir, f"gif_ep{ep}_w{w}.gif")
                    ani.save(gif_path, dpi=100, writer=PillowWriter(fps=5))
                    print(f'Saved gif at {gif_path}')

        # Save final model
        if save_model and ep == n_epoch - 1:
            model_path = os.path.join(save_dir, f"model_{ep}.pth")
            torch.save(ddpm.state_dict(), model_path)
            print(f'Saved model at {model_path}')

    # Plot training loss curve
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, n_epoch + 1), epoch_losses, marker='o')
    plt.title("DDPM Training Loss per Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Average MSE Loss")
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, "loss_curve.png"))
    plt.show()


Tasks to complete: completely rework training function for my project
Maybe start with just generating the waveguides, then move to generating waveguides and parameters

In [ ]:

if __name__ == "__main__":
    train_waveguide2(dataset)


Epoch 1/50


loss: 0.0252:   5%|▌         | 181/3501 [00:49<15:02,  3.68it/s]


KeyboardInterrupt: 